# Test 013 - Brain vs LLM: data-scaling run (50M / 100M / 500M)

Runs the no-backprop cerebellar brain at three data budgets on 2x T4, coherent dual-GPU (averages every block), and prints a WikiText-103 scoreboard after each.

## Do this before running
1. Right sidebar: **Accelerator = GPU T4 x2**, **Internet = On**.
2. **Add-ons -> Secrets**: add a secret named **GITHUB_TOKEN** = a GitHub PAT that can read `PlangoDev/plango-labs` (Contents: Read).
3. To run unattended while you're away (recommended): **Save Version -> Save & Run All (Commit)**. The run continues on Kaggle's servers after you close the tab; read the results in the version's log.

Approx wall-clock: 50M ~25 min, 100M ~50 min, 500M ~4 h. The 500M token cache is already built from your earlier run; 50M and 100M build their own (prefix) caches first, a few minutes each. Each run prints its own SCOREBOARD at the end.

In [ ]:
# 1. dependencies
!pip -q install -U transformers datasets

In [ ]:
# 2. clone the repo (private) using the GITHUB_TOKEN secret
import os
from kaggle_secrets import UserSecretsClient
os.environ["GH_TOKEN"] = UserSecretsClient().get_secret("GITHUB_TOKEN")
os.chdir("/kaggle/working")
!rm -rf plango-labs
!git clone https://$GH_TOKEN@github.com/PlangoDev/plango-labs.git
os.chdir("/kaggle/working/plango-labs/test_013")
print("cwd:", os.getcwd())

In [ ]:
# 3. 50M tokens (coherent dual-GPU)
!python showdown.py --train-tokens 50000000 --sync-every 1

In [ ]:
# 4. 100M tokens
!python showdown.py --train-tokens 100000000 --sync-every 1

In [ ]:
# 5. 500M tokens (uses the prebuilt cache)
!python showdown.py --train-tokens 500000000 --sync-every 1

## Reading the results
Each of the three runs ends with a SCOREBOARD block: LLM perplexity, Brain perplexity, and MACs/token for each. Compare the Brain perplexity across 50M -> 100M -> 500M to see whether more data closes the gap. If a run hits a VRAM OOM, change its line to add `--n-gran 32768`.